In [1]:
from typing import Any, Dict, List, Optional, Tuple
from config import RAGConfig, DEFAULT_CONFIG
import logging
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_classic.retrievers import BM25Retriever
import torch
from langchain_text_splitters import MarkdownHeaderTextSplitter
import uuid
import hashlib
from pathlib import Path
import json
import sys
import chromadb

logger = logging.getLogger(__name__)

logger.setLevel(logging.INFO)

if not logger.hasHandlers():
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.DEBUG)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)


COLLECTION_NAME = "knowledge_base"


class VectorStoreBuilder:
    def __init__(
        self,
        model_name: str = "BAAI/bge-small-zh-v1.5",
        index_save_path: str = "./vector_index",
    ):
        self.model_name = model_name
        self.index_save_path = index_save_path
        self.embeddings = None
        self.vectorstore: Optional[Chroma] = None
        self._client: Optional[chromadb.ClientAPI] = None
        self.setup_embeddings()

    def setup_embeddings(self):
        """Setup embeddings"""
        logger.info(f"Initializing embedding model: {self.model_name}")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.embeddings = HuggingFaceEmbeddings(
            model_name=self.model_name,
            model_kwargs={"device": device},
            encode_kwargs={"normalize_embeddings": True},
        )

        logger.info("Embedding model initialized successfully")

    def _get_client(self) -> chromadb.ClientAPI:
        """
        Get the ChromaDB client.

        Returns:
            ChromaDB client
        """
        if not self._client:
            Path(self.index_save_path).mkdir(parents=True, exist_ok=True)
            self._client = chromadb.PersistentClient(path=self.index_save_path)
        return self._client

    def _get_vectorstore(self) -> Chroma:
        """
        Get the Chroma vector store.

        Returns:
            Chroma vector store
        """
        if not self.vectorstore:
            client = self._get_client()
            self.vectorstore = Chroma(
                client=client,
                collection_name=COLLECTION_NAME,
                embedding_function=self.embeddings,
            )
        return self.vectorstore

    def collection_exists(self) -> bool:
        """
        Check if the collection exists in the vector store.

        Returns:
            True if the collection exists, False otherwise
        """
        try:
            client = self._get_client()
            collection = client.get_collection(name=COLLECTION_NAME)
            return collection.count() > 0
        except Exception:
            return False

    def build_vector_index(self, chunks: List[Document]) -> Chroma:
        """
        Build a Chroma vector index from the provided document chunks.

        Args:
            chunks: List of Document objects to be indexed

        Returns:
            Chroma vector store
        """
        logger.info("Building Chroma vector index...")

        if not chunks:
            raise ValueError("Document chunk list cannot be empty")

        self.reset_collection()

        client = self._get_client()
        # Build Chroma vector store
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            client=client,
            collection_name=COLLECTION_NAME,
        )

        logger.info(f"Vector index built successfully with {len(chunks)} documents")
        return self.vectorstore

    def add_documents(self, new_chunks: List[Document]):
        """
        Add new documents to the existing index

        Args:
            new_chunks: List of new document chunks
        """
        if not new_chunks:
            raise ValueError("New document chunk list cannot be empty")

        vectorstore = self._get_vectorstore()
        logger.info(f"Adding {len(new_chunks)} new documents to the index...")
        vectorstore.add_documents(new_chunks)
        logger.info("New documents added successfully")

    def delete_documents_by_source(self, sources: List[str]):
        """
        Delete documents from the index based on their source

        Args:
            sources: List of document sources to be deleted
        """
        if not sources:
            raise ValueError("Source list cannot be empty")

        client = self._get_client()
        try:
            collection = client.get_collection(name=COLLECTION_NAME)
        except Exception as e:
            logger.error(f"Error deleting documents: {e}")
            return

        for source in sources:
            results = collection.get(where={"source": source})
            if results and results["ids"]:
                collection.delete(ids=results["ids"])
                logger.info(
                    f"Deleted {len(results['ids'])} documents with source: {source}"
                )

        self.vectorstore = None

    def get_all_documents(self) -> List[Document]:
        """
        Retrieve all documents from the index

        Returns:
            List of all documents in the index
        """
        client = self._get_client()

        try:
            collection = client.get_collection(name=COLLECTION_NAME)
        except Exception as e:
            logger.error(f"Error retrieving documents: {e}")
            return []

        results = collection.get(include=["documents", "metadatas"])
        documents = []
        for doc_text, metadata in zip(results["documents"], results["metadatas"]):
            documents.append(Document(page_content=doc_text, metadata=metadata))

        logger.info(f"Retrieved {len(documents)} documents from the index")
        return documents

    def get_vectorstore(self) -> Chroma:
        """
        Get the vector store object

        Returns:
            Vector store object
        """
        return self._get_vectorstore()

    def reset_collection(self):
        """
        Reset the vector store collection
        """
        client = self._get_client()
        try:
            client.delete_collection(name=COLLECTION_NAME)
        except Exception as e:
            logger.error(f"Error resetting collection: {e}")
            pass
        self.vectorstore = None

    def similarity_search(self, query: str, k: int = 5) -> List[Document]:
        """
        Similarity search

        Args:
            query: Query text
            k: Number of results to return

        Returns:
            List of similar documents
        """
        vectorstore = self._get_vectorstore()

        return vectorstore.similarity_search(query, k=k)


d:\Code\tr-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class DocumentChunker:
    """
    Data preparation class - responsible for knowledge base data loading, cleaning, and preprocessing for RAG system.
    """

    def __init__(self, data_path: str) -> None:
        """
        Initialize data preparation class.

        Args:
            data_path: Root directory path of the knowledge base.
        """
        self.data_path = data_path
        self.documents: List[Document] = []
        self.chunks: List[Document] = []
        self.parent_child_map: Dict[str, str] = {}

    def load_documents(self, file_paths: Optional[List[str]] = None) -> List[Document]:
        """
        Load documents from the knwoledge base directory

        Args:
            file_paths: List of file paths to load. If None, all files in the data directory will be loaded.

        Returns:
            List of loaded documents
        """
        logger.info(f"Loading documents from {self.data_path}")

        documents = []
        data_path_obj = Path(self.data_path)

        if file_paths is not None:
            md_files = [data_path_obj / fp for fp in file_paths]
        else:
            md_files = list(data_path_obj.rglob("*.md"))

        for md_file in md_files:
            try:
                with open(file=md_file, mode="r", encoding="utf-8") as f:
                    content = f.read()

                try:
                    data_root = Path(self.data_path).resolve()
                    relative_path = (
                        Path(md_file).resolve().relative_to(data_root).as_posix()
                    )
                except Exception:
                    relative_path = Path(md_file).as_posix()

                parent_id = hashlib.md5(relative_path.encode("utf-8")).hexdigest()

                doc = Document(
                    page_content=content,
                    metadata={
                        "source": str(md_file),
                        "parent_id": parent_id,
                        "doc_type": "parent",
                    },
                )

                documents.append(doc)

            except Exception as e:
                logger.error(f"Error reading file {md_file}: {e}")

        for doc in documents:
            self._enhace_metadata(doc)

        self.documents = documents
        logger.info(f"Successfully loaded {len(documents)} documents")
        return documents

    def _enhace_metadata(self, doc: Document) -> None:
        """
        Enhance document metadata - extract category information from file path hierarchy

        Args:
            doc: Document to enhance metadata for
        """
        file_path = Path(doc.metadata.get("source", ""))
        data_root = Path(self.data_path).resolve()

        try:
            relative = file_path.resolve().relative_to(data_root)
            path_hierarchy = list(relative.parent.parts)
        except ValueError, RuntimeError:
            path_hierarchy = []

        doc.metadata["doc_name"] = file_path.stem
        doc.metadata["path_hierarchy"] = path_hierarchy
        doc.metadata["primary_category"] = (
            path_hierarchy[0] if len(path_hierarchy) >= 1 else ""
        )
        doc.metadata["subcategory"] = (
            path_hierarchy[1] if len(path_hierarchy) >= 2 else ""
        )

    def chunk_documents(self) -> List[Document]:
        """
        Markdown structure-aware chunking

        Returns:
            List of chunked documents
        """
        logger.info("Performing Markdown structure-aware chunking...")

        if not self.documents:
            raise ValueError("Please load documents first.")

        chunks = self._markdown_header_split()

        for i, chunk in enumerate(chunks):
            if "chunk_id" not in chunk.metadata:
                chunk.metadata["chunk_id"] = str(uuid.uuid4())
            chunk.metadata["batch_index"] = i
            chunk.metadata["chunk_size"] = len(chunk.page_content)

        self.chunks = chunks
        logger.info(f"Markdown chunking complete, generated {len(chunks)} chunks")
        return chunks

    def _markdown_header_split(self) -> List[Document]:
        """
        Perform structured splitting using Markdown header splitter

        Returns:
            List of documents split by header structure
        """
        headers_to_split_on = [
            ("#", "h1"),
            ("##", "h2"),
            ("###", "h3"),
        ]

        markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=headers_to_split_on,
            strip_headers=False,
        )

        all_chunks = []

        for doc in self.documents:
            try:
                content_preview = doc.page_content[:200]
                has_headers = any(
                    line.strip().startswith("#") for line in content_preview.split("\n")
                )

                if not has_headers:
                    logger.warning(
                        f"Document {doc.metadata.get('doc_name', 'unknown')} has no Markdown headers"
                    )
                    logger.debug(f"Content preview: {content_preview}")

                md_chunks = markdown_splitter.split_text(doc.page_content)

                logger.debug(
                    f"Document {doc.metadata.get('doc_name', 'unknown')} split into {len(md_chunks)} chunks"
                )

                if len(md_chunks) <= 1:
                    logger.warning(
                        f"Document {doc.metadata.get('doc_name', 'unknown')} could not split by headers, may lack header structure"
                    )

                parent_id = doc.metadata["parent_id"]

                for i, chunk in enumerate(md_chunks):
                    child_id = str(uuid.uuid4())
                    chunk.metadata.update(doc.metadata)
                    chunk.metadata.update(
                        {
                            "chunk_id": child_id,
                            "parent_id": parent_id,
                            "doc_type": "child",
                            "chunk_index": i,
                        }
                    )

                    self.parent_child_map[child_id] = parent_id

                all_chunks.extend(md_chunks)

            except Exception as e:
                logger.warning(
                    f"Markdown splitting failed for {doc.metadata.get('source', 'unknown')}: {e}"
                )
                all_chunks.append(doc)

        logger.info(
            f"Markdown structure splitting complete, generated {len(all_chunks)} structured chunks"
        )
        return all_chunks

    def filter_documents(self, field: str, value: str) -> List[Document]:
        """
        Filter documents by metadata field.

        Args:
            field: Metadata field name (e.g. primary_category, subcategory, doc_name)
            value: Expected field Value

        Returns:
            Filtered list of documents
        """
        return [doc for doc in self.documents if doc.metadata.get(field) == value]

    def get_statistics(self) -> Dict[str, Any]:
        """
        Get data statistics

        Returns:
            Dictionary of statistics
        """
        if not self.documents:
            return {}

        primary_categories: Dict[str, int] = {}
        subcategories: Dict[str, int] = {}

        for doc in self.documents:
            pc = doc.metadata.get("primary_category", "")
            sc = doc.metadata.get("subcategory", "")
            primary_categories[pc] = primary_categories.get(pc, 0) + 1
            if sc:
                subcategories[sc] = subcategories.get(sc, 0) + 1

        return {
            "total_documents": len(self.documents),
            "total_chunks": len(self.chunks),
            "primary_categories": primary_categories,
            "subcategories": subcategories,
            "avg_chunk_size": sum(
                chunk.metadata.get("chunk_size", 0) for chunk in self.chunks
            )
            / len(self.chunks)
            if self.chunks
            else 0,
        }

    @staticmethod
    def get_statistics_from_chunks(chunks: List[Document]) -> Dict[str, Any]:
        """
        Get statistics from a list of document chunks.

        Args:
            chunks: List of document chunks

        Returns:
            Dictionary of statistics
        """
        if not chunks:
            return {}

        parent_ids = set()
        primary_categories: Dict[str, int] = {}
        subcategories: Dict[str, int] = {}

        for chunk in chunks:
            parent_id = chunk.metadata.get("parent_id", "")
            if parent_id:
                parent_ids.add(parent_id)

            pc = chunk.metadata.get("primary_category", "")
            sc = chunk.metadata.get("subcategory", "")
            if pc:
                primary_categories[pc] = primary_categories.get(pc, 0) + 1
            if sc:
                subcategories[sc] = subcategories.get(sc, 0) + 1

        return {
            "total_documents": len(parent_ids),
            "total_chunks": len(chunks),
            "primary_categories": primary_categories,
            "subcategories": subcategories,
            "avg_chunk_size": sum(
                chunk.metadata.get("chunk_size", 0) for chunk in chunks
            )
            / len(chunks)
            if chunks
            else 0,
        }

    def export_metadata(self, output_path: str):
        """
        Export metadata to a JSON file.

        Args:
            output_path: Output file path
        """

        metadata_list = []
        for doc in self.documents:
            metadata_list.append(
                {
                    "source": doc.metadata.get("source"),
                    "doc_name": doc.metadata.get("doc_name"),
                    "primary_category": doc.metadata.get("primary_category"),
                    "subcategory": doc.metadata.get("subcategory"),
                    "path_hierarchy": doc.metadata.get("path_hierarchy"),
                    "content_length": len(doc.page_content),
                }
            )

        with open(file=output_path, mode="w", encoding="utf-8") as f:
            json.dump(metadata_list, f, ensure_ascii=False, indent=2)

        logger.info(f"Metadata exported to: {output_path}")

    def get_parent_documents(self, child_chunks: List[Document]) -> List[Document]:
        """
        Retrieves parent documents from child chunks (with smart deduplication).

        Args:
            child_chunks: List of retrieved child chunks

        Returns:
            Deduplicated list of parent documents, sorted by relevance
        """
        parent_relevance = {}
        parent_docs_map = {}

        for chunk in child_chunks:
            parent_id = chunk.metadata.get("parent_id")
            if parent_id:
                parent_relevance[parent_id] = parent_relevance.get(parent_id, 0) + 1

                if parent_id not in parent_docs_map:
                    for doc in self.documents:
                        if doc.metadata.get("parent_id") == parent_id:
                            parent_docs_map[parent_id] = doc
                            break

        sorted_parent_ids = sorted(
            parent_relevance.keys(), key=lambda x: parent_relevance[x], reverse=True
        )

        parent_docs = []
        for parent_id in sorted_parent_ids:
            if parent_id in parent_docs_map:
                parent_docs.append(parent_docs_map[parent_id])

        parent_info = []
        for doc in parent_docs:
            doc_name = doc.metadata.get("doc_name", "unknown")
            parent_id = doc.metadata.get("parent_id")
            relevance_count = parent_relevance.get(parent_id, 0)
            parent_info.append(f"{doc_name}({relevance_count}块)")

        logger.info(
            f"Found {len(child_chunks)} deduplicated parent documents from {len(parent_docs)} child chunks: {', '.join(parent_info)}"
        )
        return parent_docs


In [3]:
class RetrievalOptimization:
    """Retrieval Optimization"""

    def __init__(self, vectorstore: Chroma, chunks: List[Document]):
        """
        Initialize the RetrievalOptimization class.

        Args:
            vectorstore: Chroma vector store
            chunks: List of documents
        """
        self.vectorstore = vectorstore
        self.chunks = chunks
        self.setup_retrievers()

    def setup_retrievers(self):
        """Setup the retrievers for both vector search and BM25 search"""
        logger.info("Setting up retrievers...")

        self.vector_retriever = self.vectorstore.as_retriever(
            search_type="similarity", search_kwargs={"k": 5}
        )

        self.bm25_retriever = BM25Retriever.from_documents(self.chunks, k=5)

        logger.info("Retrievers setup completed")

    def hybrid_search(self, query: str, top_k: int = 3) -> List[Document]:
        """
        Hybrid search - combine vector search and BM25 search, use RRF for reranking

        Args:
            query: Query text
            top_k: Number of results to return

        Returns:
            List of retrieved documents
        """

        vector_docs = self.vector_retriever.invoke(query)
        bm25_docs = self.bm25_retriever.invoke(query)

        reranked_docs = self._rrf_rerank(vector_docs, bm25_docs)
        return reranked_docs[:top_k]

    def metadata_filtered_search(
        self, query: str, filters: Dict[str, Any], top_k: int = 5
    ) -> List[Document]:
        """
        Metadata-filtered search

        Args:
            query: Query text
            filters: Metadata filter conditions
            top_k: Number of results to return

        Returns:
            Filtered list of documents
        """

        docs = self.hybrid_search(query, top_k * 3)

        filtered_docs = []
        for doc in docs:
            match = True
            for key, value in filters.items():
                if key in doc.metadata:
                    if isinstance(value, list):
                        if doc.metadata[key] not in value:
                            match = False
                            break
                    else:
                        if doc.metadata[key] != value:
                            match = False
                            break
                else:
                    match = False
                    break

            if match:
                filtered_docs.append(doc)
                if len(filtered_docs) >= top_k:
                    break

        return filtered_docs

    def _rrf_rerank(
        self, vector_docs: List[Document], bm25_docs: List[Document], k: int = 60
    ) -> List[Document]:
        """
        Re-rank documents using RRF (Reciprocal Rank Fusion) algorithm

        Args:
            vector_docs: Vector search results
            bm25_docs: BM25 search results
            k: RRF parameter, used for smoothing rankings

        Returns:
            Re-ranked list of documents
        """
        doc_scores = {}
        doc_objects = {}

        for rank, doc in enumerate(vector_docs):
            doc_id = hash(doc.page_content)
            doc_objects[doc_id] = doc

            rrf_score = 1.0 / (k + rank + 1)
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + rrf_score

            logger.debug(
                f"Vector search - Document {rank + 1}: RRF score = {rrf_score:.4f}"
            )

        for rank, doc in enumerate(bm25_docs):
            doc_id = hash(doc.page_content)
            doc_objects[doc_id] = doc

            rrf_score = 1.0 / (k + rank + 1)
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + rrf_score

            logger.debug(
                f"BM25 search - Document {rank + 1}: RRF score = {rrf_score:.4f}"
            )

        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)

        reranked_docs = []
        for doc_id, final_score in sorted_docs:
            if doc_id in doc_objects:
                doc = doc_objects[doc_id]
                doc.metadata["rrf_score"] = final_score
                reranked_docs.append(doc)
                logger.debug(
                    f"Final ranking - Document: {doc.page_content[:50]}... Final RRF score: {final_score:.4f}"
                )

        logger.info(
            f"RRF re-ranking completed: Vector search {len(vector_docs)} documents, BM25 search {len(bm25_docs)} documents, merged {len(reranked_docs)} documents"
        )

        return reranked_docs


In [4]:
class ChangeDetector:
    def __init__(self, hash_file_path: str) -> None:
        self.hash_file_path = Path(hash_file_path)

    def compute_file_hashes(self, data_path: str) -> Dict[str, str]:
        data_root = Path(data_path).resolve()
        hashes = {}

        for md_file in data_root.rglob("*.md"):
            try:
                relative_path = md_file.resolve().relative_to(data_root).as_posix()
                file_path = self._compute_md5(md_file)
                hashes[relative_path] = file_path
            except Exception as e:
                logger.error(f"Error processing file {md_file}: {e}")

        return hashes

    def detect_changes(self, data_path: str) -> Tuple[List[str], List[str], List[str]]:
        current_hashes = self.compute_file_hashes(data_path)
        saved_hashes = self.load_hashes()

        added = []
        modified = []
        deleted = []

        for path, hash_val in current_hashes.items():
            if path not in saved_hashes:
                added.append(path)
            elif saved_hashes[path] != hash_val:
                modified.append(path)

        for path in saved_hashes:
            if path not in current_hashes:
                deleted.append(path)

        logger.info(
            f"Change detection results - Added: {added}, Modified: {modified}, Deleted: {deleted}"
        )
        return added, deleted, modified

    def load_hashes(self) -> Dict[str, str]:
        if not self.hash_file_path.exists():
            return {}
        try:
            with open(self.hash_file_path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception as e:
            logger.error(f"Error loading hashes from {self.hash_file_path}: {e}")
            return {}

    def save_hashes(self, data_path: str) -> None:
        hashes = self.compute_file_hashes(data_path)
        self.hash_file_path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.hash_file_path, "w", encoding="utf-8") as f:
            json.dump(hashes, f, ensure_ascii=False, indent=2)

        logger.info(f"Saved hashes to {self.hash_file_path}")

    def clear_hashes(self) -> None:
        if self.hash_file_path.exists():
            self.hash_file_path.unlink()
            logger.info(f"Cleared hashes from {self.hash_file_path}")

    @staticmethod
    def _compute_md5(file_path: Path) -> str:
        md5 = hashlib.md5()
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(8192), b""):
                md5.update(chunk)
        return md5.hexdigest()


In [6]:
class RAGSystem:
    def __init__(self, config: Optional[RAGConfig] = None):
        """
        Initialize the RAG system.

        Args:
            config: Configuration for the RAG system.
        """
        logger.info("🚀 Initializing RAG system...")

        self.config: RAGConfig = config or DEFAULT_CONFIG
        self.retrieval_optimization: Optional[RetrievalOptimization] = None

        if not Path(self.config.knowledge_base_path).exists():
            raise FileNotFoundError(
                f"Knowledge base path does not exist: {self.config.knowledge_base_path}"
            )
        logger.info("Initializing DocumentChunker...")
        self.doc_chunker = DocumentChunker(self.config.knowledge_base_path)

        logger.info("Initializing VectorStoreBuilder...")
        self.vector_builder = VectorStoreBuilder(
            model_name=self.config.embedding_model,
            index_save_path=self.config.index_save_path,
        )

        hash_file = str(Path(self.config.index_save_path) / "file_hashes.json")
        self.change_detector = ChangeDetector(hash_file)

        logger.info("✅ RAG system initialization complete.")

    def build_knowledge_base(self) -> None:
        """
        Build the knowledge base by chunking documents, building an index, and saving the index.
        """
        if self.vector_builder.collection_exists():
            logger.info("Detecting knowledge base changes...")

            added, modified, deleted = self.change_detector.detect_changes(
                self.config.knowledge_base_path
            )

            if added or modified or deleted:
                logger.info(
                    "Changes detected: Added: {len(added)}, Modified: {len(modified)}, Deleted: {len(deleted)}"
                )
                self._incremental_update(added, modified, deleted)
            else:
                logger.info("No changes detected.")

            logger.info("✅ Successfully loaded existing index.")
            chunks = self.vector_builder.get_all_documents()
        else:
            logger.info(
                "No existing index found. Building a new index from the knowledge base."
            )
            chunks = self._full_build()

        logger.info("✅ Successfully built knowledge base.")
        vectorstore = self.vector_builder.get_vectorstore()
        self.retrieval_optimization = RetrievalOptimization(vectorstore, chunks)

        stats = DocumentChunker.get_statistics_from_chunks(chunks)
        logger.info(f"Knowledge base statistics: {stats}")
        logger.info(f"Documents count: {stats['total_documents']}")
        logger.info(f"Chunks count: {stats['total_chunks']}")
        logger.info(f"Primary categories: {list(stats['primary_categories'].keys())}")
        logger.info(f"Subcategories: {stats['subcategories']}")

        logger.info("✅ Knowledge base construction complete.")

    def _full_build(self) -> List[Document]:
        """
        Perform a full build of the knowledge base.

        Returns:
            List of document chunks used for building the index
        """
        self.doc_chunker.load_documents()
        chunks = self.doc_chunker.chunk_documents()
        self.vector_builder.build_vector_index(chunks)
        self.change_detector.save_hashes(self.config.knowledge_base_path)
        return chunks

    def _incremental_update(
        self, added: List[str], modified: List[str], deleted: List[str]
    ) -> None:
        """
        Perform an incremental update of the knowledge base based on detected changes.

        Args:
            added: List of added file paths
            modified: List of modified file paths
            deleted: List of deleted file paths
        """
        data_path = Path(self.config.knowledge_base_path)

        if deleted:
            sources_to_delete = [str(data_path / fp) for fp in deleted]
            logger.info(f"Deleted {len(deleted)} index for deleted documents")
            self.vector_builder.delete_documents_by_source(sources_to_delete)

        if modified:
            sources_to_delete = [str(data_path / fp) for fp in modified]
            logger.info(f"Updated {len(modified)} index for modified documents")
            self.vector_builder.delete_documents_by_source(sources_to_delete)

        files_to_process = added + modified
        if files_to_process:
            logger.info(f"Processing {len(files_to_process)} documents")
            self.doc_chunker.load_documents(file_paths=files_to_process)
            new_chunks = self.doc_chunker.chunk_documents()
            self.vector_builder.add_documents(new_chunks)

        self.change_detector.save_hashes(self.config.knowledge_base_path)

    def search(self, query: str, top_k: int = 5) -> List[Document]:
        """
        Search the knowledge base with a query.

        Args:
            query: Query text
            top_k: Number of results to return

        Returns:
            List of retrieved documents
        """
        if not self.retrieval_optimization:
            raise ValueError("Please build the knowledge base first")

        return self.retrieval_optimization.hybrid_search(query, top_k=top_k)


# query = "FAISS是做什么的？"
# query = "How to perform inner join with SQLAlchemy in Python?"
query = "天然珠光云母水解过程PH值？"
rag = RAGSystem()

rag.build_knowledge_base()

results = rag.search(query, top_k=5)

logger.info(f"User query: '{query}'")
logger.info("Similarity search results:")
for i, doc in enumerate(results):
    logger.info(f"{i + 1}. {doc.page_content}")


2026-05-30 00:08:35,842 - INFO - 🚀 Initializing RAG system...
2026-05-30 00:08:35,843 - INFO - Initializing DocumentChunker...
2026-05-30 00:08:35,844 - INFO - Initializing VectorStoreBuilder...
2026-05-30 00:08:35,845 - INFO - Initializing embedding model: D:/modelscope/hub/models/microsoft/harrier-oss-v1-0___6b


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6032.73it/s]


2026-05-30 00:08:40,030 - INFO - Embedding model initialized successfully
2026-05-30 00:08:40,031 - INFO - ✅ RAG system initialization complete.
2026-05-30 00:08:40,038 - INFO - Detecting knowledge base changes...
2026-05-30 00:08:40,055 - INFO - Change detection results - Added: [], Modified: [], Deleted: []
2026-05-30 00:08:40,056 - INFO - No changes detected.
2026-05-30 00:08:40,057 - INFO - ✅ Successfully loaded existing index.
2026-05-30 00:08:40,067 - INFO - Retrieved 92 documents from the index
2026-05-30 00:08:40,068 - INFO - ✅ Successfully built knowledge base.
2026-05-30 00:08:40,072 - INFO - Setting up retrievers...
2026-05-30 00:08:40,076 - INFO - Retrievers setup completed
2026-05-30 00:08:40,077 - INFO - Knowledge base statistics: {'total_documents': 8, 'total_chunks': 92, 'primary_categories': {'technologies': 85, 'manufacture': 7}, 'subcategories': {'langchain': 21, 'python': 36, 'rag': 19, 'sqlalchemy': 9, 'mica': 7}, 'avg_chunk_size': 325.80434782608694}
2026-05-30 00